# rl_torch

> Phase 12, the PyTorch side of `clue_rl`: log-probabilities of reply tokens (left padding, so only the reply positions' logits are computed) and checkpoints (adapters, optimizer state, step, configuration hash) with refusal of another configuration's checkpoint.

In [ ]:
#| default_exp rl_torch

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pathlib import Path

from explRL.local_policy import encodePair, promptText


def replyLogps(model, tok, samples, maxLen, device):
    "Per sample, the log-probabilities of its reply tokens (left padding, so the replies end the batch together)."
    import torch
    seqs = [encodePair(tok, promptText(tok, s['system'], s['blocks']), s['reply'], maxLen, s['eos']) for s in samples]
    L, K = max(len(x) for x, _ in seqs), max(len(x) - st for x, st in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for n, (x, _) in enumerate(seqs):
        ids[n, L - len(x):], mask[n, L - len(x):] = torch.tensor(x), 1
    ids, mask = ids.to(device), mask.to(device)
    pos = (mask.cumsum(-1) - 1).clamp(min=0)
    logits = model(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=K + 1).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, L - K:, None])[..., 0]
    return [lp[n, K - (len(x) - st):] for n, (x, st) in enumerate(seqs)]


def saveMemory(model, gradCkpt=True):
    "Gradient checkpointing (non-reentrant, so LoRA training works): activations are recomputed in the backward pass."
    if gradCkpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=dict(use_reentrant=False))
    return model


def policyUpdate(model, tok, opt, params, samples, chunks, names, perRole, kl, maxLen, device):
    """One policy-gradient step over `samples` (in micro-batches `chunks`): mean over samples of -A * mean log pi(reply)
    + kl * mean k3 to the same model with its adapters off; gradient clipping 1.0. -> loss, KL, tokens. If the step runs
    out of GPU memory without gradient checkpointing, checkpointing is switched on and the step is redone."""
    import gc
    import torch
    oom = getattr(torch, 'OutOfMemoryError', torch.cuda.OutOfMemoryError)
    try:
        return _update(model, tok, opt, params, samples, chunks, names, perRole, kl, maxLen, device)
    except oom:
        if getattr(model, 'is_gradient_checkpointing', True):
            raise
    # retried outside the except block: inside it, the traceback keeps the failed attempt's tensors alive
    opt.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    saveMemory(model, True)
    print('out of GPU memory: gradient checkpointing switched on; the step is redone')
    return _update(model, tok, opt, params, samples, chunks, names, perRole, kl, maxLen, device)


def _update(model, tok, opt, params, samples, chunks, names, perRole, kl, maxLen, device):
    import torch
    stats = dict(loss=0.0, kl=0.0, tokens=0)
    for chunk in chunks:
        if perRole:
            model.set_adapter(names[chunk[0]['slot']])
        with torch.no_grad(), model.disable_adapter():
            ref = replyLogps(model, tok, chunk, maxLen, device)
        lps = replyLogps(model, tok, chunk, maxLen, device)
        loss = 0.0
        for x, lp, rf in zip(chunk, lps, ref):
            d = rf - lp
            k3 = (torch.exp(d) - d - 1).mean()
            loss = loss - x['adv'] * lp.mean() + kl * k3
            stats['kl'] += k3.item() / len(samples)
            stats['tokens'] += len(lp)
        (loss / len(samples)).backward()
        stats['loss'] += loss.item() / len(samples)
    if samples:
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
    opt.zero_grad()
    return stats


def _ckpts(out):
    return sorted(Path(out).glob('ckpt-*'))


def saveCheckpoint(model, opt, step, out, config, ro=None):
    """out/ckpt-<step>: the adapters, the optimizer state, the step and the run's configuration hash; with the step's
    rollout `ro`, also its turn logs (out/turns/step-<step>.jsonl), to read what the agents do."""
    import torch
    d = Path(out) / f'ckpt-{step:04d}'
    model.save_pretrained(d)
    torch.save(opt.state_dict(), d / 'optimizer.pt')
    (d / 'state.json').write_text(json.dumps(dict(step=step, config=config)) + '\n')
    if ro is not None:
        (Path(out) / 'turns').mkdir(exist_ok=True)
        (Path(out) / 'turns' / f'step-{step:04d}.jsonl').write_text(''.join(
            json.dumps(dict(board=b, seed=ro.seeds[b], **x), sort_keys=True) + '\n'
            for b in range(len(ro.boards)) for lg in ro.logs[b] for x in lg))
    return d


def loadCheckpoint(model, opt, out, config, names):
    "Resume from the newest checkpoint: -> its step (0 without one); refuses one written by another configuration."
    import torch
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    if not _ckpts(out):
        return 0
    d = _ckpts(out)[-1]
    st = json.loads((d / 'state.json').read_text())
    if st['config'] != config:
        raise ValueError(f'{d} was written by another configuration ({st["config"]} != {config})')
    for name in names:
        set_peft_model_state_dict(model, load_file(d / ('' if name == 'default' else name) / 'adapter_model.safetensors'),
                                  adapter_name=name)
    opt.load_state_dict(torch.load(d / 'optimizer.pt'))
    return st['step']

## Tests

In [ ]:
import inspect
from explRL.rl_torch import *
# the GPU-side pieces need the [train] extra; their behaviour is checked by clue_rl's train cell and the Colab pilot
assert list(inspect.signature(loadCheckpoint).parameters) == ['model', 'opt', 'out', 'config', 'names']
assert 'logits_to_keep' in inspect.getsource(replyLogps)       # only the reply positions' logits are computed

In [ ]:
#| train
# With the [train] extra, no download: an out-of-memory step without gradient checkpointing switches it on and is redone
import tempfile
import torch
import explRL.rl_torch as RT
from peft import get_peft_model
from explRL.clue_sft import loadModel, loraConfig
from explRL.tiny_model import tinyModel
with tempfile.TemporaryDirectory() as d:
    tok, base = loadModel(tinyModel(d), 'cpu')
    model = get_peft_model(base, loraConfig(4, 8))
    params = [p for n, p in model.named_parameters() if 'lora_' in n]
    opt = torch.optim.AdamW(params, lr=1e-3)
    sample = dict(slot=0, system='rules', blocks=[dict(type='text', text='Now:\nobs')], reply='{"action": "wait"}', eos=True, adv=1.0)
    _real, calls = RT._update, []
    def fake(*a):
        calls.append(model.is_gradient_checkpointing)
        if len(calls) == 1:
            raise getattr(torch, 'OutOfMemoryError', torch.cuda.OutOfMemoryError)('simulated')
        return _real(*a)
    RT._update = fake                     # policyUpdate looks _update up in the module
    try:
        assert not model.is_gradient_checkpointing
        st = RT.policyUpdate(model, tok, opt, params, [sample], [[sample]], ['default'], False, 0.04, 512, 'cpu')
        assert calls == [False, True] and model.is_gradient_checkpointing and st['tokens'] > 0
    finally:
        RT._update = _real

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()